In [6]:
from scripts.functions import *
import yaml
import plotly.express as px

# Read in YAML containing file paths
file_paths = yaml.safe_load(open("./utilities/file_paths.yaml", "r"))

# Read in dataset
name_data = pl.read_parquet(file_paths["parquet_year_path"])
name_data_state = pl.read_parquet(file_paths["parquet_state_path"])

ModuleNotFoundError: No module named 'scripts.functions'

Exploratory Analysis
    
    - Number of Records: 2.2M (375m names, 189m Male, 185m Female)
    
    - Number of Years Captured: 146
    
    - Total Distinct Names: 106k (45k Males, 72k Females, 12k Shared Names)

Data Quality Issues:

In [ ]:
# Number of Distinct Male and Female Names by year
names_per_year = (
    name_data
    .group_by("year", "sex")
    .agg(pl.col("name").n_unique().alias("n_names"))
    .sort("year")
)

px.bar(
    names_per_year,
    x="year",
    y="n_names",
    color="sex",
    barmode="group",
    title="Number of Distinct Names by Year and Sex",
    labels={"year": "Year", "n_names": "Number of Distinct Names", "sex": "Sex"},
)

In [2]:
# Get distinct records
name_data.select(
    pl.all()
    .n_unique()
)

name,sex,count,year
u32,u32,u32,u32
105966,2,14006,146


In [ ]:
# Names used for both Male and Female
male_names = name_data.filter(pl.col("sex") == "M").get_column("name").unique()
female_names = name_data.filter(pl.col("sex") == "F").get_column("name").unique()
shared_names = male_names.to_frame().join(female_names.to_frame(), on="name", how="inner")

print(f"Male Names: {male_names.count()} | Female Names: {female_names.count()} | Shared Names: {shared_names.height}")

Male Names: 45481 | Female Names: 72339 | Shared Names: 11854


In [6]:
# Most Common Names by Sex
most_common_names = (
    name_data
    .group_by("sex", "name")
    .agg(pl.col("count").sum())
    .sort("count", descending=True)
    .group_by("sex")
    .agg(pl.all().first())
)

most_common_names.head()

sex,name,count
str,str,i64
"""F""","""Mary""",4141481
"""M""","""James""",5250638


Number of names used for both sexes: 11854


In [12]:
# Most Common Names by Sex and Year
most_common_by_year = (
    name_data
    .sort("count", descending=True)
    .group_by("year", "sex")
    .agg(pl.all().first())
    .sort("year", "sex")
)

most_common_by_year

year,sex,name,count
i32,str,str,i64
1880,"""F""","""Mary""",7065
1880,"""M""","""John""",9655
1881,"""F""","""Mary""",6919
1881,"""M""","""John""",8768
1882,"""F""","""Mary""",8148
…,…,…,…
2023,"""M""","""Liam""",20904
2024,"""F""","""Olivia""",14772
2024,"""M""","""Liam""",22252


In [30]:
# Racing bar chart: Top 10 names by year with sex filter
import plotly.graph_objects as go
from scripts.functions import get_top_names_by_year

# Get top 10 names for each sex
top_n = 10
male_data = get_top_names_by_year(
    name_data.filter(pl.col("sex") == "M"), 
    rank=1, 
    top_n=top_n
)
female_data = get_top_names_by_year(
    name_data.filter(pl.col("sex") == "F"), 
    rank=1, 
    top_n=top_n
)

# Add sex labels
male_data = male_data.with_columns(pl.lit("Male").alias("sex_label"))
female_data = female_data.with_columns(pl.lit("Female").alias("sex_label"))

# Combine data
combined_data = pl.concat([male_data, female_data])
years = sorted(combined_data.get_column("year").unique())

# Create initial figure with male data
initial_data = male_data.filter(pl.col("year") == years[0]).sort("rank")

fig = go.Figure(
    data=[go.Bar(
        x=initial_data.get_column("count"),
        y=initial_data.get_column("name"),
        text=initial_data.get_column("name"),
        orientation="h",
        marker=dict(color="#1f77b4"),
        textposition="inside",
        textfont=dict(color="white", size=14),
        hovertemplate="<b>%{y}</b><br>Count: %{x:,}<extra></extra>"
    )]
)

# Create frames for both sexes
all_frames = []

for sex_label in ["Male", "Female"]:
    sex_data = combined_data.filter(pl.col("sex_label") == sex_label)
    color = "#1f77b4" if sex_label == "Male" else "#ff7f0e"
    
    for year in years:
        year_data = sex_data.filter(pl.col("year") == year).sort("rank")
        
        all_frames.append(go.Frame(
            data=[go.Bar(
                x=year_data.get_column("count"),
                y=year_data.get_column("name"),
                text=year_data.get_column("name"),
                orientation="h",
                marker=dict(color=color),
                textposition="inside",
                textfont=dict(color="white", size=14),
                hovertemplate="<b>%{y}</b><br>Count: %{x:,}<extra></extra>"
            )],
            name=f"{sex_label.lower()}_year{year}",
            layout={"sliders": [{"active": years.index(year)}]}
        ))

fig.frames = all_frames

# Create sex filter dropdown buttons
sex_buttons = []
for sex_label in ["Male", "Female"]:
    button = dict(
        label=sex_label,
        method="animate",
        args=[
            [f"{sex_label.lower()}_year{years[0]}"],
            {
                "mode": "immediate",
                "frame": {"duration": 0, "redraw": True},
                "transition": {"duration": 0}
            }
        ]
    )
    sex_buttons.append(button)

# Create slider steps that reference both male and female frames
slider_steps = []
for i, year in enumerate(years):
    slider_steps.append({
        "args": [
            # This will be ignored by the animation, but required by slider structure
            [f"male_year{year}"],
            {
                "frame": {"duration": 150, "redraw": True},
                "mode": "immediate",
                "transition": {"duration": 50}
            }
        ],
        "method": "animate",
        "label": str(year)
    })

# Update layout
fig.update_layout(
    updatemenus=[
        # Play/Pause buttons
        dict(
            type="buttons",
            buttons=[
                dict(
                    label="▶ Play",
                    method="animate",
                    args=[None, {
                        "frame": {"duration": 150, "redraw": True},
                        "fromcurrent": True,
                        "transition": {"duration": 50},
                        "mode": "immediate"
                    }]
                ),
                dict(
                    label="⏸ Pause",
                    method="animate",
                    args=[[None], {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "transition": {"duration": 0}
                    }]
                )
            ],
            direction="left",
            pad={"r": 10, "t": 10},
            showactive=False,
            x=0.02,
            xanchor="left",
            y=1.15,
            yanchor="top"
        ),
        # Sex dropdown on the right
        dict(
            type="dropdown",
            buttons=sex_buttons,
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            active=0,
            x=0.98,
            xanchor="right",
            y=1.15,
            yanchor="top"
        )
    ],
    title="Top 10 Most Popular Baby Names by Year",
    xaxis_title="Number of Babies",
    yaxis_title="",
    xaxis=dict(range=[0, combined_data.get_column("count").max() * 1.1]),
    yaxis=dict(autorange="reversed"),  # Rank 1 at top
    showlegend=False,
    height=600,
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "y": -0.05,
        "xanchor": "left",
        "currentvalue": {
            "prefix": "Year: ",
            "visible": True,
            "xanchor": "right"
        },
        "pad": {"b": 10, "t": 50},
        "len": 0.9,
        "x": 0.1,
        "steps": slider_steps
    }]
)

fig.show()

In [1]:
# Probability of having a given name
def compute_name_probability(name: str, sex: str = None, year: int = None) -> dict:
    """
    Compute the probability someone will have a given name.
    
    Args:
        name (str): The name to look up
        sex (str): Optional - 'M' or 'F' to filter by sex
        year (int): Optional - specific year to analyze
    
    Returns:
        dict: Dictionary with probability and supporting statistics
    """
    # Filter data
    filtered = name_data.filter(pl.col("name") == name)
    
    if sex:
        filtered = filtered.filter(pl.col("sex") == sex)
    
    if year:
        filtered = filtered.filter(pl.col("year") == year)
    
    # Calculate statistics
    name_count = filtered.get_column("count").sum()
    
    # Get total for comparison
    comparison_data = name_data
    if sex:
        comparison_data = comparison_data.filter(pl.col("sex") == sex)
    if year:
        comparison_data = comparison_data.filter(pl.col("year") == year)
    
    total_count = comparison_data.get_column("count").sum()
    
    # Calculate probability
    probability = (name_count / total_count) if total_count > 0 else 0
    
    return {
        "name": name,
        "sex": sex if sex else "All",
        "year": year if year else "All Years",
        "count": name_count,
        "total": total_count,
        "probability": probability,
        "percentage": probability * 100,
        "odds": f"1 in {int(1/probability):,}" if probability > 0 else "N/A"
    }

# Example: Probability of being named "James"
result = compute_name_probability("James", sex="M")
print(f"Name: {result['name']}")
print(f"Sex: {result['sex']}")
print(f"Period: {result['year']}")
print(f"Total with name: {result['count']:,}")
print(f"Total births: {result['total']:,}")
print(f"Probability: {result['probability']:.6f} ({result['percentage']:.4f}%)")
print(f"Odds: {result['odds']}")

print("\n" + "="*60 + "\n")

# Example: Probability of being named "Olivia" in 2024
result_2024 = compute_name_probability("Olivia", sex="F", year=2024)
print(f"Name: {result_2024['name']}")
print(f"Sex: {result_2024['sex']}")
print(f"Period: {result_2024['year']}")
print(f"Total with name: {result_2024['count']:,}")
print(f"Total births: {result_2024['total']:,}")
print(f"Probability: {result_2024['probability']:.6f} ({result_2024['percentage']:.4f}%)")
print(f"Odds: {result_2024['odds']}")

NameError: name 'name_data' is not defined